# Travailler avec les modèles de la famille Meta 

## Introduction 

Cette leçon couvrira : 

- Exploration des deux principaux modèles de la famille Meta - Llama 3.1 et Llama 3.2, ainsi que des alternatives actuellement disponibles utilisées dans les exemples de code (Llama 3.3 et Llama 4) 
- Compréhension des cas d'utilisation et des scénarios pour chaque modèle 
- Exemple de code pour montrer les fonctionnalités uniques de chaque modèle 


## La famille de modèles Meta 

Dans cette leçon, nous allons explorer 2 modèles de la famille Meta ou "Troupeau Llama" - Llama 3.1 et Llama 3.2. Les exemples de code exécutables utilisent Llama 3.3 et Llama 4, qui sont des alternatives actuellement disponibles dans le catalogue Microsoft Foundry Models. 

Ces modèles existent en différentes variantes et sont disponibles dans le [catalogue Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Note :** GitHub Models sera arrêté à la fin de juillet 2026. Voici plus de détails sur l'utilisation de [Microsoft Foundry Models](https://learn.microsoft.com/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) pour prototyper avec des modèles d'IA.

> **Note :** Les versions spécifiques des modèles pour lesquelles cette leçon a été initialement écrite (Llama 3.1 70B/405B Instruct, Llama 3.2 11B/90B Vision Instruct) ont depuis été retirées du catalogue Microsoft Foundry Models. Les exemples de code ci-dessous utilisent `Llama-3.3-70B-Instruct` et `Llama-4-Scout-17B-16E-Instruct` comme alternatives actuellement disponibles dans le catalogue Microsoft Foundry Models, ils fonctionnent donc toujours aujourd’hui.

Variantes du modèle (comme couvert initialement dans cette leçon) : 
- Llama 3.1 - 70B Instruct (retiré) 
- Llama 3.1 - 405B Instruct (retiré) 
- Llama 3.2 - 11B Vision Instruct (retiré) 
- Llama 3.2 - 90B Vision Instruct (retiré) 

Alternatives actuellement disponibles utilisées dans les exemples de code : 
- Llama 3.3 - 70B Instruct (`Llama-3.3-70B-Instruct`) 
- Llama 4 Scout - 17B 16E Instruct (`Llama-4-Scout-17B-16E-Instruct`) 

*Note : Les variantes Llama 3 discutées ci-dessus sont conservées à titre historique et ne sont plus disponibles dans Microsoft Foundry Models.*



## Llama 3.1 

Avec 405 milliards de paramètres, Llama 3.1 appartient à la catégorie des LLM open source. 

Ce modèle est une amélioration par rapport à la version précédente Llama 3, en offrant : 

- Fenêtre de contexte plus grande - 128k tokens contre 8k tokens 
- Nombre maximal de tokens en sortie plus élevé - 4096 contre 2048 
- Meilleur support multilingue - dû à l'augmentation du nombre de tokens d'entraînement 

Cela permet à Llama 3.1 de gérer des cas d'utilisation plus complexes lors de la création d'applications GenAI, notamment : 
- Appels de fonctions natifs - la capacité d'appeler des outils et fonctions externes en dehors du flux de travail du LLM
- Meilleure performance RAG - grâce à une fenêtre de contexte plus étendue 
- Génération de données synthétiques - la capacité de créer des données efficaces pour des tâches comme l'ajustement fin 



### Appel de fonction native 

Llama 3.1 a été affiné pour être plus efficace dans l’exécution d’appels de fonctions ou d’outils. Il dispose également de deux outils intégrés que le modèle peut identifier comme nécessitant d’être utilisés en fonction de l’invite de l’utilisateur. Ces outils sont : 

- **Brave Search** - Peut être utilisé pour obtenir des informations à jour comme la météo en effectuant une recherche web 
- **Wolfram Alpha** - Peut être utilisé pour des calculs mathématiques plus complexes, ce qui évite d’écrire vos propres fonctions. 

Vous pouvez également créer vos propres outils personnalisés que le LLM pourra appeler. 

Dans l’exemple de code ci-dessous : 

- Nous définissons les outils disponibles (brave_search, wolfram_alpha) dans l’invite système. 
- Envoyez une invite utilisateur qui demande la météo dans une certaine ville. 
- Le LLM répondra avec un appel de fonction vers l’outil Brave Search qui ressemblera à ceci `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Note : Cet exemple effectue uniquement l’appel à l’outil, si vous souhaitez obtenir les résultats, vous devrez créer un compte gratuit sur la page de l’API Brave et définir la fonction elle-même` 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Meta-Llama-3.1-405B-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-3.3-70B-Instruct, a currently available alternative, instead
model_name = "Llama-3.3-70B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Bien que Llama 3.1 soit un LLM, une limitation qu'il présente est la multimodalité. C'est-à-dire la capacité à utiliser différents types d'entrées comme des images comme invites et à fournir des réponses. Cette capacité est l'une des principales caractéristiques de Llama 3.2. Ces fonctionnalités incluent également : 

- Multimodalité - a la capacité d'évaluer à la fois des invites textuelles et des images 
- Variations de taille petite à moyenne (11B et 90B) - cela offre des options de déploiement flexibles, 
- Variations uniquement textuelles (1B et 3B) - cela permet de déployer le modèle sur des appareils edge / mobiles et offre une faible latence 

Le support multimodal représente une avancée majeure dans le monde des modèles open source. L'exemple de code ci-dessous prend à la fois une image et une invite textuelle pour obtenir une analyse de l'image. Comme Llama 3.2 90B Vision Instruct a été retiré, l'exemple utilise `Llama-4-Scout-17B-16E-Instruct`, une alternative native multimodale actuellement disponible. 

### Support multimodal avec Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
# Llama-3.2-90B-Vision-Instruct was retired from Microsoft Foundry on 2026-06-13 with no replacement listed;
# using Llama-4-Scout-17B-16E-Instruct, a currently available natively multimodal alternative, instead
model_name = "Llama-4-Scout-17B-16E-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## L'apprentissage ne s'arrête pas ici, continuez votre parcours

Après avoir terminé cette leçon, consultez notre [collection d'apprentissage sur l’IA générative](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) pour continuer à améliorer vos connaissances en IA générative !


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
